In [ ]:
from __future__ import annotations

import argparse
import json
import os
import tempfile
from dataclasses import asdict, dataclass, field
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pywt
from skimage.color import rgb2gray
from skimage import exposure
from skimage.feature import graycomatrix, graycoprops
from skimage.io import imread
from skimage.transform import resize
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from steelblast_classic_features import CLASS_NAMES, load_split

# SteelBlastQC Dataset
The SteelBlastQC dataset contains steel surfaces that are either shot-blasted or still need shot-blasting to achieve the required texture. The dataset includes “good” images (ready for paint) images and 766 “not-good” images (needs shot-blasting) images. As declared by the collaborating manufacturer, the ideally treated surface is clean and uniformly coarse with an average roughness level of SA 2.5. The “not-good” class presents several types of defects to the surface, located by industrial shot-blasting experts. These include: fresh welding lines and cuts, abrasion, corrosion, and discoloration.

# Load Dataset
This step reads training and test images from the structured dataset folders, collects file paths, and generates numeric labels for each image class.
It does not load image arrays immediately; instead it creates a lightweight image path list that is later used for preprocessing, normalization, feature extraction, and classification.


In [ ]:
@dataclass(frozen=True)
class AppConfig:
    dataset_dir: Path = Path("data/SteelBlastQC")
    output_model: Path = Path("steelblast_svm_glcm_dwt.joblib")
    metrics_json: Path = Path("steelblast_svm_glcm_dwt_metrics.json")

app_config = AppConfig()

train_paths, y_train = load_split(app_config.dataset_dir, "train")
test_paths, y_test = load_split(app_config.dataset_dir, "test")




print(f"Loaded {len(train_paths)} training images and {len(test_paths)} test images.")

# `GLCM` + `DWT` feature extraction
Handcrafted texture features are data-efficient and interpretable for small/industrial image sets; combining co-occurrence (GLCM) and multi-scale decomposition (DWT) captures complementary texture information while avoiding deep-learning data/compute demands.
- **Input & Resize**: `image_size = 256` — preserves texture detail while bounding computational cost and standardizing feature dimensions for GLCM/DWT.

- **Color & Illumination**: grayscale conversion with `illumination_normalization = "percentile"` and `contrast_percentiles = (2.0, 98.0)`.
  This setting was adopted after robustness recovery experiments because it improved high-contrast stability and restored the historical robustness target.
  TODO compare this approach to clahe

- **GLCM Design**:
    - `glcm_levels = 32` — limits matrix size and noise sensitivity while retaining characteristic texture gradients.
    - `glcm_distances = (1,2,4,8)` and `glcm_angles = (0, π/4, π/2, 3π/4)` — multi-scale, multi-orientation sampling captures varied blast textures.
    - `glcm_properties = ("contrast","dissimilarity","homogeneity","energy","correlation","ASM")` — summarize local variation, uniformity, and correlation.

- **DWT Design**:
    - `dwt_wavelet = "db2"`, `dwt_level = 3` — Daubechies wavelets provide compact time-frequency localization; three levels capture coarse-to-fine texture without exploding feature count.
    - `describe_coefficients(...)` uses mean, std, absolute moments, energy, entropy, and percentiles to robustly summarize coefficient distributions.

- **Feature Composition & Preprocessing**: concatenate GLCM + DWT features; `StandardScaler` standardizes features because SVMs depend on distances. `quantize_image(...)` stabilizes GLCM inputs.


In [ ]:
from steelblast_classic_features import (
    FeatureExtractionConfig,
    extract_features_from_image,
    load_split,
    normalize_illumination,
    read_grayscale_image,
)


@dataclass(frozen=True)
class BrightnessAugmentationConfig:
    enabled: bool = True
    # Include identity plus mild/moderate brightening around the bright_25 stress level.
    factors: tuple[float, ...] = (1.0, 1.10, 1.20, 1.25)


# Loops through all images
# Extracts features
# Stacks them into matrix:
def build_feature_matrix(
    image_paths: list[Path],
    config: FeatureExtractionConfig,
    split_name: str,
    brightness_factors: tuple[float, ...] = (1.0,),
) -> np.ndarray:
    features: list[np.ndarray] = []
    total = len(image_paths) * len(brightness_factors)
    processed = 0

    for image_path in image_paths:
        for brightness_factor in brightness_factors:
            features.append(extract_features(image_path, config, brightness_factor))
            processed += 1
            if processed % 100 == 0 or processed == total:
                print(f"{split_name}: extracted {processed}/{total} feature rows")

    return np.vstack(features)


def balanced_limit(
    image_paths: list[Path],
    labels: np.ndarray,
    limit_per_class: int,
) -> tuple[list[Path], np.ndarray]:
    selected_paths: list[Path] = []
    selected_labels: list[int] = []

    for label in sorted(np.unique(labels)):
        class_indices = np.flatnonzero(labels == label)[:limit_per_class]
        selected_paths.extend(image_paths[index] for index in class_indices)
        selected_labels.extend(labels[index] for index in class_indices)

    return selected_paths, np.asarray(selected_labels, dtype=np.int64)


def extract_features(
    image_path: Path,
    config: FeatureExtractionConfig,
    brightness_factor: float = 1.0,
) -> np.ndarray:
    #
    image = read_grayscale_image(image_path, config.image_size, config=None)
    image = np.clip(image * float(brightness_factor), 0.0, 1.0).astype(np.float32)
    image = normalize_illumination(image, config)
    return extract_features_from_image(image, config)


feature_config = FeatureExtractionConfig(
    image_size=256, # image size to resize to (256x256)
    illumination_normalization="percentile", # illumination normalization method
    contrast_percentiles=(2.0, 98.0),
    glcm_levels=32, # Number of gray levels for GLCM quantization
    glcm_properties=("contrast",
                     "dissimilarity",
                       "homogeneity",
                         "energy",
                           "correlation", "ASM"), # GLCM properties to compute
    dwt_wavelet="db2", # Wavelet type for DWT
    dwt_level=3 # Decomposition level for DWTs
)
brightness_aug_config = BrightnessAugmentationConfig(
    enabled=True,
    factors=(1.0, 1.10, 1.20, 1.25),
)

train_brightness_factors = (
    brightness_aug_config.factors if brightness_aug_config.enabled else (1.0,)
)

print(f"Train images: {len(train_paths)}")
print(f"Test images:  {len(test_paths)}")
print(f"Feature extraction config: {feature_config}")
print(f"Train brightness factors: {train_brightness_factors}")

import time
preprocess_start = time.perf_counter()

#TODO # Split train into train and validation
train_features_start = time.perf_counter()
X_train = build_feature_matrix(
    train_paths,
    feature_config,
    "train",
    brightness_factors=train_brightness_factors,
)
train_feature_extraction_seconds = float(time.perf_counter() - train_features_start)
test_features_start = time.perf_counter()
X_test = build_feature_matrix(
    test_paths,
    feature_config,
    "test",
    brightness_factors=(1.0,),
)
test_feature_extraction_seconds = float(time.perf_counter() - test_features_start)
y_train_model = np.repeat(y_train, len(train_brightness_factors)).astype(np.int64)
preprocessing_time_seconds = float(time.perf_counter() - preprocess_start)
print(f"Augmented training labels: {y_train_model.shape}")
print(f"Train feature extraction time (s): {train_feature_extraction_seconds:.2f}")
print(f"Test feature extraction time (s): {test_feature_extraction_seconds:.2f}")
print(f"Total preprocessing time (s): {preprocessing_time_seconds:.2f}")
 

# Modelling and Evaluation

## Model selection
- Pipeline: `StandardScaler -> CalibratedClassifierCV(SVC(kernel="rbf", class_weight="balanced"))`
- Selection objective: `scoring="f1"`
- Outer CV splitter: `StratifiedKFold` with controlled fold count based on class support.

## Training and Validation Approach
1. Use **Stratified K-fold Cross-Validation** inside `GridSearchCV` to select the best SVM hyperparameters (`C`, `gamma`) by mean CV F1 score.
2. **Refit** the selected pipeline on the **entire training dataset** (`GridSearchCV` default `refit=True`).
3. Evaluate the refit model **once** on the held-out **test set** (`X_test`, `y_test`).

### CV-Tuned Hyperparameters
- `svm__estimator__C`: `[0.1, 1, 10, 100]`
- `svm__estimator__gamma`: `["scale", 0.01, 0.001, 0.0001]`
- Optimized metric: mean cross-validated `f1`

### Best found hyper parameters
- Best parameters: {'svm__estimator__C': 100, 'svm__estimator__gamma': 'scale'}
- Best CV F1: 0.9987

## Parameters
- `svm__estimator__C`
  - Regularization strength inverse. Lower values regularize more; higher values allow tighter decision boundaries.
- `svm__estimator__gamma`
  - RBF kernel influence radius. Larger values make more local boundaries; smaller values make smoother/global boundaries.
- `max_cv_folds`
  - Upper cap for outer CV folds; actual folds are reduced when class counts are small.
- `calibrated_cv_folds`
  - Number of folds used by `CalibratedClassifierCV` to learn sigmoid probability calibration.
- `scoring`
  - Metric optimized during hyperparameter search (`f1`), emphasizing balanced precision/recall.
- `class_weight="balanced"`
  - Reweights classes inversely to frequency to reduce majority-class bias.
- `n_jobs`
  - Parallelism level for `GridSearchCV`; increase when resources permit.
- `random_state`
  - Seed used in CV shuffling for reproducible splits.


In [ ]:
@dataclass(frozen=True)
class TrainingConfig:
    param_grid: dict[str, list[object]] = field(
        default_factory=lambda: {
            "svm__estimator__C": [0.1, 1, 10, 100],
            "svm__estimator__gamma": ["scale", 0.01, 0.001, 0.0001],
        }
    )
    # 5-fold CV is a standard bias-variance compromise for small/medium datasets:
    # it provides stable hyperparameter ranking versus 3-fold, while keeping runtime
    # far lower than 10-fold. The effective folds are capped by the smallest class
    # count (see `outer_splits = min(max_cv_folds, min_class_count)`) to preserve
    # stratification feasibility and avoid invalid splits on minority classes.
    max_cv_folds: int = 5
    scoring: str = "f1"
    random_state: int = 42
    calibrated_cv_folds: int = 3
    # Parallel jobs for GridSearchCV. Use -1 outside restricted environments.
    n_jobs: int = 1
    verbose: int = 2


training_config = TrainingConfig()

# train model using SVM (RBF kernel)
def train_model(X_train: np.ndarray, y_train: np.ndarray, config: TrainingConfig) -> GridSearchCV:
    min_class_count = int(np.bincount(y_train).min())
    outer_splits = min(config.max_cv_folds, min_class_count)
    if outer_splits < 2:
        raise ValueError("Each class needs at least two training images.")

    calibration_splits = min(config.calibrated_cv_folds, min_class_count)
    if calibration_splits < 2:
        raise ValueError("Each class needs at least two training images for probability calibration.")

    base_svm = SVC(kernel="rbf", class_weight="balanced")
    calibrated_svm = CalibratedClassifierCV(
        estimator=base_svm,
        method="sigmoid",
        cv=calibration_splits,
        ensemble=False, #Use a single calibrated model instead of an ensemble of models
    )
    pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()), # StandardScaler standardizes features by removing the mean and scaling to unit variance, which is important for SVM performance since it relies on distances in feature space. This ensures that all features contribute equally to the model and prevents features with larger scales from dominating the decision boundary.
            ("svm", calibrated_svm), # Use an explicitly calibrated SVM so predict_proba remains available after sklearn removes SVC(probability=True).
        ]
    )
    # Stratified splitting keeps the same class proportions in every fold as in the whole dataset.
    # random_state ensures reproducibility, and shuffle=True randomizes the data before splitting to avoid any order bias.
    cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=config.random_state)
    # GridSearchCV exhaustively tries all combinations of the specified hyperparameters (C and gamma for the SVM) and evaluates each combination using cross-validation on the training data. It selects the combination that yields the best average F1 score across the folds.
    search = GridSearchCV(
        pipeline,
        param_grid=config.param_grid,
        scoring=config.scoring,
        cv=cv,
        n_jobs=config.n_jobs,
        verbose=config.verbose,
    )
    search.fit(X_train, y_train)
    return search


def run_inference(trained_model: GridSearchCV, X_eval: np.ndarray, y_eval: np.ndarray):
    inference_start = time.perf_counter()
    y_pred_local = trained_model.predict(X_eval)
    inference_time_seconds_local = float(time.perf_counter() - inference_start)
    inference_samples_local = int(len(y_eval))
    inference_time_per_sample_ms_local = float((inference_time_seconds_local * 1000.0) / max(inference_samples_local, 1))
    return y_pred_local, inference_time_seconds_local, inference_samples_local, inference_time_per_sample_ms_local


def evaluate_predictions(y_true: np.ndarray, y_pred_local: np.ndarray, class_names: list[str]):
    report_local = classification_report(
        y_true,
        y_pred_local,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    matrix_local = confusion_matrix(y_true, y_pred_local).tolist()
    return report_local, matrix_local


print(f"Training feature matrix: {X_train.shape}")
print(f"Training labels:         {y_train_model.shape}")
print(f"Testing feature matrix:  {X_test.shape}")

import time
import ast
import inspect
import textwrap


def _estimate_cyclomatic_complexity(fn) -> int | None:
    """Estimate cyclomatic complexity via a lightweight AST decision-point count."""
    try:
        src = textwrap.dedent(inspect.getsource(fn))
        tree = ast.parse(src)
    except Exception:
        return None

    complexity = 1
    for node in ast.walk(tree):
        if isinstance(node, (ast.If, ast.For, ast.AsyncFor, ast.While, ast.IfExp, ast.ExceptHandler, ast.Assert, ast.Match)):
            complexity += 1
        elif isinstance(node, ast.BoolOp):
            complexity += max(len(node.values) - 1, 0)
        elif isinstance(node, ast.comprehension):
            complexity += len(node.ifs)
    return complexity


pipeline_cyclomatic_complexity = {
    'build_feature_matrix': _estimate_cyclomatic_complexity(build_feature_matrix),
    'balanced_limit': _estimate_cyclomatic_complexity(balanced_limit),
    'extract_features': _estimate_cyclomatic_complexity(extract_features),
    'train_model': _estimate_cyclomatic_complexity(train_model),
    'run_inference': _estimate_cyclomatic_complexity(run_inference),
    'evaluate_predictions': _estimate_cyclomatic_complexity(evaluate_predictions),
}
pipeline_cyclomatic_complexity['total'] = int(sum(v for v in pipeline_cyclomatic_complexity.values() if isinstance(v, int)))
print(f"Estimated pipeline cyclomatic complexity total: {pipeline_cyclomatic_complexity['total']}")
training_start = time.perf_counter()
model = train_model(X_train, y_train_model, training_config)
training_time_seconds = float(time.perf_counter() - training_start)
print(f"Training time (s): {training_time_seconds:.2f}")


# Evaluate the model on the test set
y_pred, inference_time_seconds, inference_samples, inference_time_per_sample_ms = run_inference(
    model,
    X_test,
    y_test,
)
print(f"Inference time (s): {inference_time_seconds:.4f}")
print(f"Inference time per sample (ms): {inference_time_per_sample_ms:.4f}")

report, matrix = evaluate_predictions(y_test, y_pred, CLASS_NAMES)
joblib.dump(model.best_estimator_, app_config.output_model)

print(f"Best parameters: {model.best_params_}")
print(f"Best CV F1: {model.best_score_:.4f}")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))
print("Confusion matrix:")
print(np.asarray(matrix))

model_svm = model
test_paths_svm = test_paths #path of images used for testing
y_test_svm = y_test #array of labels
X_test_svm = X_test #feature matrix of test images
y_pred_svm = y_pred #array of predicted labels
%store model_svm
%store test_paths_svm
%store y_test_svm
%store X_test_svm
%store y_pred_svm
%store feature_config




In [ ]:
metrics = {
    "best_params": model.best_params_ if hasattr(model, "best_params_") else None,
    "best_cv_f1": float(model.best_score_) if hasattr(model, "best_score_") else None,
    "classification_report": report if "report" in globals() else None,
    "confusion_matrix": matrix if "matrix" in globals() else None,
    "train_images": len(train_paths) if "train_paths" in globals() else None,
    "train_feature_rows": int(X_train.shape[0]) if "X_train" in globals() else None,
    "train_brightness_factors": list(train_brightness_factors) if "train_brightness_factors" in globals() else [1.0],
    "test_images": len(test_paths),
    "feature_count": int(X_train.shape[1]) if "X_train" in globals() else int(X_test.shape[1]),
    "feature_extraction_config": asdict(feature_config),
    "preprocessing_time_seconds": float(preprocessing_time_seconds) if "preprocessing_time_seconds" in globals() else None,
    "train_feature_extraction_seconds": float(train_feature_extraction_seconds) if "train_feature_extraction_seconds" in globals() else None,
    "test_feature_extraction_seconds": float(test_feature_extraction_seconds) if "test_feature_extraction_seconds" in globals() else None,
    "training_time_seconds": float(training_time_seconds) if "training_time_seconds" in globals() else None,
    "inference_time_seconds": float(inference_time_seconds) if "inference_time_seconds" in globals() else None,
    "inference_samples": int(inference_samples) if "inference_samples" in globals() else None,
    "inference_time_per_sample_ms": float(inference_time_per_sample_ms) if "inference_time_per_sample_ms" in globals() else None,
    "cyclomatic_complexity": pipeline_cyclomatic_complexity if "pipeline_cyclomatic_complexity" in globals() else None,
    "training_config": asdict(training_config) if "training_config" in globals() else None,
}
app_config.metrics_json.write_text(json.dumps(metrics, indent=2))

print(f"Saved model to:   {app_config.output_model.resolve()}")
print(f"Saved metrics to: {app_config.metrics_json.resolve()}")